# Credit Cluster Scoring Tool

Use this notebook to score a private, simulated, or competitor company against the saved credit clustering model.

Expected workflow:

1. Put the saved artifact at `saved_models/credit_cluster_model_v1.joblib`.
2. Enter company financials manually, or load a CSV/Excel file.
3. Run the scoring cells.
4. Review cluster assignment, near-default affinity, feature coverage, and warning flags.

Important: the output is **not a formal credit rating** and not a calibrated probability of default. It is a public-company financial profile match based on distance to the learned KMeans credit clusters.

In [1]:
#Import your libraries here

import numpy as np
import pandas as pd
import joblib
from pathlib import Path
import sys

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

## 1. Load saved fitted model artifact

The training notebook should have saved this file after fitting the KMeans model.

In [2]:
# Import project modules and load trained artifact

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.utils import (
    build_credit_report_tables,
    save_credit_report_outputs,
    save_credit_pdf_report,
)

from src.credit_clustering import (
    # Artifact utilities
    load_artifact,
    get_segment_artifact,
    infer_near_default_cluster_from_artifact,

    # Scoring and diagnostics
    score_companies,
    add_adjacent_bucket_distances_and_outlook,
    compare_to_cluster_profile,
    make_scenarios,

    #Analyst guardrails
    apply_credit_guardrails,

    # Config
    DEFAULT_PRIMARY_SEGMENT,
    DEFAULT_SCORING_TEMPERATURE,
    DEFAULT_FX_TO_MODEL_CURRENCY,
    DEFAULT_SCORING_MIN_DENOMINATOR,
    RATIO_COLS,
    SUMMARY_COLS_WITH_OUTLOOK,
    SCENARIO_SUMMARY_COLS,
)

MODEL_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "saved_models"
    / "nonfinancial_credit_scorecard_kmeans_k5_v3.joblib"
)

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Model artifact not found at {MODEL_PATH}")

INPUT_PATH = PROJECT_ROOT / "inputs"
INPUT_PATH.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = PROJECT_ROOT / "outputs" / "company_scores"
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

artifact = load_artifact(MODEL_PATH)

SCORING_SEGMENT = artifact.get("primary_segment", DEFAULT_PRIMARY_SEGMENT)
SCORING_TEMPERATURE = DEFAULT_SCORING_TEMPERATURE
FX_TO_MODEL_CURRENCY = DEFAULT_FX_TO_MODEL_CURRENCY
SCORING_MIN_DENOMINATOR = DEFAULT_SCORING_MIN_DENOMINATOR

NEAR_DEFAULT_CLUSTER = infer_near_default_cluster_from_artifact(
    artifact,
    segment=SCORING_SEGMENT,
)

segment_artifact = get_segment_artifact(
    artifact,
    segment=SCORING_SEGMENT,
)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("MODEL_PATH:", MODEL_PATH)
print("Scoring segment:", SCORING_SEGMENT)
print("Artifact version:", artifact.get("artifact_version"))
print("Features used by model:", segment_artifact.get("feature_cols"))
print("Cluster labels:", segment_artifact.get("cluster_labels"))
print("Risk rank:", segment_artifact.get("risk_rank"))
print("Inferred near-default cluster:", NEAR_DEFAULT_CLUSTER)

PROJECT_ROOT: D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\machine_learning\08_final_project_3
MODEL_PATH: D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\machine_learning\08_final_project_3\outputs\saved_models\nonfinancial_credit_scorecard_kmeans_k5_v3.joblib
Scoring segment: Non-financial
Artifact version: v3_scorecard
Features used by model: ['structural_distress_risk', 'earnings_risk', 'operating_cashflow_risk', 'liquidity_risk', 'leverage_risk', 'debt_service_risk']
Cluster labels: {3: '1 - Strong relative credit profile', 0: '2 - Good credit profile', 1: '3 - Leveraged / elevated risk profile', 4: '4 - Weak credit profile', 2: '5 - Distressed / near-default proxy'}
Risk rank: {3: 1, 0: 2, 1: 3, 4: 4, 2: 5}
Inferred near-default cluster: 2


## 2. Manual input example

Edit the numbers below to score a private company, competitor, or simulated case.

Currency note: if your model was trained on USD and the input is in EUR, set `FX_TO_MODEL_CURRENCY` to the EUR/USD conversion rate. Ratios are mostly unaffected; `log_assets` is affected.

In [3]:
manual_input_2025 = pd.DataFrame([{
    "company_name": "Manual 2025 Company",
    "fiscal_year": 2025,
    "currency": "USD",
    "major_sector": "Manufacturing / Industrials",
    "financial_flag": "Non-financial",

    # Values converted from thousand BGN to full USD units at 1.7 BGN/USD
    "revenue": 48_585_294.12,
    "assets": 29_721_275.23,
    "current_assets": 10_037_745.82,
    "cash": 34_804.65,
    "receivables": 2_208_235.29,
    "inventory": 7_794_705.88,
    "equity": 9_082_352.94,
    "current_liabilities": 12_722_352.94,
    "liabilities": 20_638_823.53,
    "long_term_debt": 7_910_588.24,
    "short_term_debt": 1_478_235.29,
    "net_income": 949_411.76,
    "cfo": 3_558_235.29,
    "capex": 588_235.29,
    "operating_income": 1_664_705.88,
    "interest_expense": 597_647.06,

    # Optional EBITDA inputs used by the patched v3 scorer.
    # If direct EBITDA is not available, scoring.py can calculate it from
    # operating_income + depreciation_amortization when both are supplied.
    "depreciation_amortization": 2_713_000,
    "ebitda": np.nan,
}])

template_path = INPUT_PATH / "model_2025_company.csv"

manual_input_2025.to_csv(template_path, index=False)
print("Template saved to:", template_path)


Template saved to: D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\machine_learning\08_final_project_3\inputs\model_2025_company.csv


In [4]:
eeec_input_2025 = pd.DataFrame([{
    "company_name": "Eastern European Electric Company B.V.",
    "fiscal_year": 2025,
    "currency": "BGN",
    "major_sector": "Transportation / Utilities",
    "financial_flag": "Non-financial",
    "revenue": 2_572_207_000,
    "assets": 2_127_723_000,
    "current_assets": 1_024_267_000,
    "cash": 224_558_000,
    "receivables": 465_950_000,
    "inventory": 42_958_000,
    "equity": 457_686_000,
    "current_liabilities": 541_002_000,
    "liabilities": 1_670_037_000,
    "long_term_debt": 988_887_000,
    "short_term_debt": 14_753_000,
    "net_income": 36_799_000,
    "cfo": 265_267_000,
    "capex": 212_690_000,
    "operating_income": 54_961_000,
    "interest_expense": 102_302_000,

    # Optional EBITDA inputs used by the patched v3 scorer.
    # If direct EBITDA is not available, scoring.py can calculate it from
    # operating_income + depreciation_amortization when both are supplied.
    "depreciation_amortization": 119_519_000,
    "ebitda": 259_574_000,
}])

template_path = INPUT_PATH / "EEEC_2025.csv"

eeec_input_2025.to_csv(template_path, index=False)
print("Template saved to:", template_path)

Template saved to: D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\machine_learning\08_final_project_3\inputs\EEEC_2025.csv


In [5]:
template_path = INPUT_PATH / "EEEC_2025.csv"

current_input = pd.read_csv(template_path)
current_input

,company_name,fiscal_year,currency,major_sector,financial_flag,revenue,assets,current_assets,cash,receivables,inventory,equity,current_liabilities,liabilities,long_term_debt,short_term_debt,net_income,cfo,capex,operating_income,interest_expense,depreciation_amortization,ebitda
0,Eastern European Electric Company B.V.,2025,BGN,Transportation / Utilities,Non-financial,2572207000,2127723000,1024267000,224558000,465950000,42958000,457686000,541002000,1670037000,988887000,14753000,36799000,265267000,212690000,54961000,102302000,119519000,259574000


In [6]:
scored_manual = score_companies(
    input_df=current_input,
    artifact=artifact,
    segment=SCORING_SEGMENT,
    temperature=SCORING_TEMPERATURE,
    fx_to_model_currency=FX_TO_MODEL_CURRENCY,
    min_denominator=DEFAULT_SCORING_MIN_DENOMINATOR,
    near_default_cluster=NEAR_DEFAULT_CLUSTER,
)

scored_manual_with_outlook = add_adjacent_bucket_distances_and_outlook(
    scored_manual,
    artifact,
    neutral_band=0.15,
    upgrade_boundary_multiplier=1.35,
    downgrade_boundary_multiplier=1.35,
)

current_external_rating = None # optional, none is not existant
scored_manual_with_outlook = apply_credit_guardrails(
    scored_manual_with_outlook,
    external_rating=current_external_rating
)

existing_summary_cols_with_outlook = [
    col for col in SUMMARY_COLS_WITH_OUTLOOK
    if col in scored_manual_with_outlook.columns
]

missing_summary_cols_with_outlook = [
    col for col in SUMMARY_COLS_WITH_OUTLOOK
    if col not in scored_manual_with_outlook.columns
]

if missing_summary_cols_with_outlook:
    print("Missing summary columns:", missing_summary_cols_with_outlook)

display(scored_manual_with_outlook[existing_summary_cols_with_outlook])

,company_name,fiscal_year,assigned_cluster,cluster_label,risk_rank,cluster_affinity,near_default_affinity,distance_to_assigned_cluster,upper_bucket_cluster,upper_bucket_label,distance_to_upper_bucket,lower_bucket_cluster,lower_bucket_label,distance_to_lower_bucket,outlook_flag,outlook_reason,scorecard_credit_score,feature_coverage_pct,warning_flags,guardrail_level,guardrail_flags,guardrail_summary,analyst_interpretation,commercial_conclusion
0,Eastern European Electric Company B.V.,2025,3,1 - Strong relative credit profile,1,0.327906,0.112928,0.441432,NaN,None,NaN,0,2 - Good credit profile,0.797873,Neutral,"Only the weaker adjacent bucket is available, ...",27.665618,1.0,interest_coverage_below_1,High caution,"elevated_debt_to_assets, elevated_debt_to_ebit...",The model assigns the company to the strongest...,The cluster assignment appears primarily suppo...,"The model output may be directionally useful, ..."


In [7]:
# ---------------------------------------------------------------------
# 8. Company ratio and diagnostic snapshot
# ---------------------------------------------------------------------

ratio_cols = artifact.get("ratio_cols", RATIO_COLS)

existing_ratio_cols = [
    col for col in ratio_cols
    if col in scored_manual.columns
]

missing_ratio_cols = [
    col for col in ratio_cols
    if col not in scored_manual.columns
]

if missing_ratio_cols:
    print("Missing ratio/diagnostic columns:", missing_ratio_cols)

company_ratio_snapshot = scored_manual[
    ["company_name"] + existing_ratio_cols
].copy()

display(company_ratio_snapshot.T.round(4))

,0
company_name,Eastern European Electric Company B.V.
log_assets,21.478318
liabilities_to_assets,0.784894
debt_to_assets,0.471697
debt_to_equity,2.192857
equity_to_assets,0.215106
cash_to_assets,0.105539
net_income_to_assets,0.017295
cfo_to_assets,0.124672
revenue_to_assets,1.208901


## 3. Compare the company to cluster medians

This helps explain why the company was assigned to a specific profile.

In [8]:
comparison_to_cluster = compare_to_cluster_profile(
    scored_manual.iloc[0],
    artifact,
)

comparison_to_cluster

,company_value,assigned_cluster_median,difference,relative_position
metric,,,,
log_assets,21.4783,21.1933,0.2850,above_cluster_median
liabilities_to_assets,0.7849,0.4538,0.3311,above_cluster_median
debt_to_assets,0.4717,0.2358,0.2359,above_cluster_median
equity_to_assets,0.2151,0.4851,-0.2700,below_cluster_median
current_ratio,1.8933,2.3660,-0.4727,below_cluster_median
quick_ratio,1.2764,1.2753,0.0010,above_cluster_median
cash_to_assets,0.1055,0.1174,-0.0119,below_cluster_median
net_income_to_assets,0.0173,0.0491,-0.0318,below_cluster_median
cfo_to_assets,0.1247,0.0938,0.0309,above_cluster_median


## 4. Scenario analysis

This section shows how a company migrates across clusters under stress cases

In [9]:
# ---------------------------------------------------------------------
# 10. Build scenario inputs
# ---------------------------------------------------------------------

scenario_input = make_scenarios(current_input.iloc[0])

scenario_input_cols = [
    "scenario",
    "assets",
    "liabilities",
    "equity",
    "cash",
    "revenue",
    "net_income",
    "cfo",
    "long_term_debt",
    "short_term_debt",
    "operating_income",
    "interest_expense",
    "depreciation_amortization",
    "ebitda",
]

existing_scenario_input_cols = [
    col for col in scenario_input_cols
    if col in scenario_input.columns
]

missing_scenario_input_cols = [
    col for col in scenario_input_cols
    if col not in scenario_input.columns
]

if missing_scenario_input_cols:
    print("Missing scenario input columns:", missing_scenario_input_cols)

display(scenario_input[existing_scenario_input_cols])

,scenario,assets,liabilities,equity,cash,revenue,net_income,cfo,long_term_debt,short_term_debt,operating_income,interest_expense,depreciation_amortization,ebitda
0,base,2127723000,1.670037e+09,457686000.0,224558000.0,2.572207e+09,36799000.0,265267000.0,9.888870e+08,14753000.0,54961000.0,102302000.0,119519000,259574000
0,revenue_down_15pct,2127723000,1.670037e+09,457686000.0,224558000.0,2.186376e+09,25759300.0,198950250.0,9.888870e+08,14753000.0,38472700.0,102302000.0,119519000,259574000
0,debt_up_25pct,2127723000,1.917259e+09,457686000.0,175113650.0,2.572207e+09,36799000.0,265267000.0,1.236109e+09,14753000.0,54961000.0,102302000.0,119519000,259574000
0,cash_burn_case,2127723000,1.670037e+09,457686000.0,112279000.0,2.572207e+09,-36799000.0,-265267000.0,9.888870e+08,14753000.0,54961000.0,102302000.0,119519000,259574000
0,near_default_stress,2127723000,2.340495e+09,-212772300.0,224558000.0,2.572207e+09,36799000.0,265267000.0,1.595792e+09,319158450.0,40920800.0,102302000.0,119519000,259574000


In [10]:
# ---------------------------------------------------------------------
# 11. Score scenarios
# ---------------------------------------------------------------------

scored_scenarios = score_companies(
    input_df=scenario_input,
    artifact=artifact,
    segment=SCORING_SEGMENT,
    temperature=SCORING_TEMPERATURE,
    fx_to_model_currency=FX_TO_MODEL_CURRENCY,
    min_denominator=SCORING_MIN_DENOMINATOR,
    near_default_cluster=NEAR_DEFAULT_CLUSTER,
)

scored_scenarios = add_adjacent_bucket_distances_and_outlook(
    scored_scenarios,
    artifact,
    neutral_band=0.15,
    upgrade_boundary_multiplier=1.35,
    downgrade_boundary_multiplier=1.35,
)

scored_scenarios = apply_credit_guardrails(
    scored_scenarios,
    external_rating=current_external_rating,
)

scenario_summary_cols = artifact.get("scenario_summary_cols", SCENARIO_SUMMARY_COLS)

# Add selected diagnostic ratios for direct scenario comparison.
scenario_diagnostic_cols = [
    "liabilities_to_assets",
    "debt_to_assets",
    "debt_to_equity",
    "equity_to_assets",
    "cfo_to_assets",
    "current_ratio",
    "quick_ratio",
    "interest_coverage",
    "fcf_to_debt",
    "ebitda",
    "ebitda_margin",
    "debt_to_ebitda",
    "net_debt_to_ebitda",
    "ebitda_interest_coverage",
    "debt_service_risk",
]

scenario_display_cols = []

for col in list(scenario_summary_cols) + scenario_diagnostic_cols:
    if col in scored_scenarios.columns and col not in scenario_display_cols:
        scenario_display_cols.append(col)

missing_scenario_cols = [
    col for col in list(scenario_summary_cols) + scenario_diagnostic_cols
    if col not in scored_scenarios.columns
]

if missing_scenario_cols:
    print("Missing scenario columns:", missing_scenario_cols)

display(scored_scenarios[scenario_display_cols])

,scenario,assigned_cluster,cluster_label,risk_rank,cluster_affinity,near_default_affinity,distance_to_assigned_cluster,upper_bucket_cluster,upper_bucket_label,distance_to_upper_bucket,lower_bucket_cluster,lower_bucket_label,distance_to_lower_bucket,outlook_flag,outlook_reason,scorecard_credit_score,feature_coverage_pct,warning_flags,guardrail_level,guardrail_flags,guardrail_summary,analyst_interpretation,commercial_conclusion,liabilities_to_assets,debt_to_assets,debt_to_equity,equity_to_assets,cfo_to_assets,current_ratio,quick_ratio,interest_coverage,fcf_to_debt,ebitda,ebitda_margin,debt_to_ebitda,net_debt_to_ebitda,ebitda_interest_coverage,debt_service_risk
0,base,3,1 - Strong relative credit profile,1,0.327906,0.112928,0.441432,NaN,None,NaN,0.0,2 - Good credit profile,0.797873,Neutral,"Only the weaker adjacent bucket is available, ...",27.665618,1.0,interest_coverage_below_1,High caution,"elevated_debt_to_assets, elevated_debt_to_ebit...",The model assigns the company to the strongest...,The cluster assignment appears primarily suppo...,"The model output may be directionally useful, ...",0.784894,0.471697,2.192857,0.215106,0.124672,1.893278,1.276350,0.537243,0.052386,259574000.0,0.100915,3.866489,3.001387,2.537331,0.652029
1,revenue_down_15pct,3,1 - Strong relative credit profile,1,0.315093,0.116433,0.498369,NaN,None,NaN,0.0,2 - Good credit profile,0.829739,Neutral,"Only the weaker adjacent bucket is available, ...",29.435037,1.0,interest_coverage_below_1,High caution,"elevated_debt_to_assets, elevated_debt_to_ebit...",The model assigns the company to the strongest...,The cluster assignment appears primarily suppo...,"The model output may be directionally useful, ...",0.784894,0.471697,2.192857,0.215106,0.093504,1.893278,1.276350,0.376070,-0.013690,259574000.0,0.118723,3.866489,3.001387,2.537331,0.718106
2,debt_up_25pct,3,1 - Strong relative credit profile,1,0.305054,0.120614,0.588129,NaN,None,NaN,0.0,2 - Good credit profile,0.854078,Neutral,"Only the weaker adjacent bucket is available, ...",33.110786,1.0,interest_coverage_below_1,High caution,"elevated_debt_to_assets, elevated_debt_to_ebit...",The model assigns the company to the strongest...,The cluster assignment appears primarily suppo...,"The model output may be directionally useful, ...",0.901085,0.587887,2.733013,0.215106,0.124672,1.893278,1.184956,0.537243,0.042033,259574000.0,0.100915,4.818902,4.144283,2.537331,0.721909
3,cash_burn_case,1,3 - Leveraged / elevated risk profile,3,0.303008,0.157456,0.481975,0.0,2 - Good credit profile,1.273883,4.0,4 - Weak credit profile,0.702223,Neutral,Company remains materially closer to its assig...,56.714265,1.0,"interest_coverage_below_1, negative_cfo_to_assets",High caution,"elevated_debt_to_assets, elevated_debt_to_ebit...",The company is assigned to 3 - Leveraged / ele...,The cluster assignment appears primarily suppo...,"The model output may be directionally useful, ...",0.784894,0.471697,2.192857,0.215106,-0.124672,1.893278,1.068811,0.537243,-0.476224,259574000.0,0.100915,3.866489,3.433938,2.537331,0.804416
4,near_default_stress,2,5 - Distressed / near-default proxy,5,0.243697,0.243697,1.282745,4.0,4 - Weak credit profile,1.651468,NaN,None,NaN,Positive,Company is near the stronger adjacent bucket a...,47.828710,1.0,"liabilities_exceed_assets, negative_equity, hi...",Override required,"elevated_debt_to_assets, high_debt_to_assets, ...",The company is assigned to the weakest model-r...,The cluster assignment appears primarily suppo...,Manual analyst override is required. The model...,1.100000,0.900000,-9.000000,-0.100000,0.124672,1.893278,1.276350,0.400000,0.027456,259574000.0,0.100915,7.377282,6.512180,2.537331,0.810304


## 5. Save report outputs to files


In [11]:
# ---------------------------------------------------------------------
# 12. Build and save credit report outputs
# ---------------------------------------------------------------------

report_tables = build_credit_report_tables(
    scored_manual_with_outlook=scored_manual_with_outlook,
    scored_manual=scored_manual,
    comparison_to_cluster=comparison_to_cluster,
    scenario_input=scenario_input,
    scored_scenarios=scored_scenarios,
    artifact=artifact,
)

current_file_name = "EEEC_2025"

saved_paths = save_credit_report_outputs(
    tables=report_tables,
    output_path=OUTPUT_PATH,
    base_filename=current_file_name,
)

print("Scores saved to:", saved_paths["score_csv"])
print("Scenario CSV saved to:", saved_paths["scenario_csv"])
print("Full Excel score report saved to:", saved_paths["report_xlsx"])

display(report_tables["scored_file"])
display(report_tables["scenario_file"])

Scores saved to: D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\machine_learning\08_final_project_3\outputs\company_scores\EEEC_2025_score_result.csv
Scenario CSV saved to: D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\machine_learning\08_final_project_3\outputs\company_scores\EEEC_2025_scenario_analysis.csv
Full Excel score report saved to: D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\machine_learning\08_final_project_3\outputs\company_scores\EEEC_2025_score_report.xlsx


,company_name,fiscal_year,assigned_cluster,cluster_label,risk_rank,cluster_affinity,near_default_affinity,distance_to_assigned_cluster,upper_bucket_cluster,upper_bucket_label,distance_to_upper_bucket,lower_bucket_cluster,lower_bucket_label,distance_to_lower_bucket,outlook_flag,outlook_reason,feature_coverage_pct,warning_flags,guardrail_level,guardrail_flags,guardrail_summary,analyst_interpretation,commercial_conclusion
0,Eastern European Electric Company B.V.,2025,3,1 - Strong relative credit profile,1,0.327906,0.112928,0.441432,NaN,None,NaN,0,2 - Good credit profile,0.797873,Neutral,"Only the weaker adjacent bucket is available, ...",1.0,interest_coverage_below_1,High caution,"elevated_debt_to_assets, elevated_debt_to_ebit...",The model assigns the company to the strongest...,The cluster assignment appears primarily suppo...,"The model output may be directionally useful, ..."


,scenario,assigned_cluster,cluster_label,risk_rank,cluster_affinity,near_default_affinity,distance_to_assigned_cluster,upper_bucket_cluster,upper_bucket_label,distance_to_upper_bucket,lower_bucket_cluster,lower_bucket_label,distance_to_lower_bucket,outlook_flag,outlook_reason,scorecard_credit_score,feature_coverage_pct,warning_flags,guardrail_level,guardrail_flags,guardrail_summary,analyst_interpretation,commercial_conclusion,log_assets,liabilities_to_assets,debt_to_assets,debt_to_equity,equity_to_assets,cash_to_assets,net_income_to_assets,cfo_to_assets,revenue_to_assets,current_ratio,quick_ratio,interest_coverage,fcf_to_debt,operating_margin,gross_margin,cfo_to_liabilities,capex_to_revenue,total_debt,net_debt,ebitda,ebitda_margin,debt_to_ebitda,net_debt_to_ebitda,ebitda_interest_coverage,leverage_risk,liquidity_risk,earnings_risk,operating_cashflow_risk,debt_service_risk,debt_service_risk_legacy,structural_distress_risk
0,base,3,1 - Strong relative credit profile,1,0.327906,0.112928,0.441432,NaN,None,NaN,0.0,2 - Good credit profile,0.797873,Neutral,"Only the weaker adjacent bucket is available, ...",27.665618,1.0,interest_coverage_below_1,High caution,"elevated_debt_to_assets, elevated_debt_to_ebit...",The model assigns the company to the strongest...,The cluster assignment appears primarily suppo...,"The model output may be directionally useful, ...",21.478318,0.784894,0.471697,2.192857,0.215106,0.105539,0.017295,0.124672,1.208901,1.893278,1.276350,0.537243,0.052386,0.021367,NaN,0.158839,0.082688,1.003640e+09,7.790820e+08,259574000.0,0.100915,3.866489,3.001387,2.537331,0.488441,0.038420,0.327050,0.0,0.652029,0.756182,0
1,revenue_down_15pct,3,1 - Strong relative credit profile,1,0.315093,0.116433,0.498369,NaN,None,NaN,0.0,2 - Good credit profile,0.829739,Neutral,"Only the weaker adjacent bucket is available, ...",29.435037,1.0,interest_coverage_below_1,High caution,"elevated_debt_to_assets, elevated_debt_to_ebit...",The model assigns the company to the strongest...,The cluster assignment appears primarily suppo...,"The model output may be directionally useful, ...",21.478318,0.784894,0.471697,2.192857,0.215106,0.105539,0.012107,0.093504,1.027566,1.893278,1.276350,0.376070,-0.013690,0.017597,NaN,0.119129,0.097280,1.003640e+09,7.790820e+08,259574000.0,0.118723,3.866489,3.001387,2.537331,0.488441,0.038420,0.378935,0.0,0.718106,0.861904,0
2,debt_up_25pct,3,1 - Strong relative credit profile,1,0.305054,0.120614,0.588129,NaN,None,NaN,0.0,2 - Good credit profile,0.854078,Neutral,"Only the weaker adjacent bucket is available, ...",33.110786,1.0,interest_coverage_below_1,High caution,"elevated_debt_to_assets, elevated_debt_to_ebit...",The model assigns the company to the strongest...,The cluster assignment appears primarily suppo...,"The model output may be directionally useful, ...",21.478318,0.901085,0.587887,2.733013,0.215106,0.082301,0.017295,0.124672,1.208901,1.893278,1.184956,0.537243,0.042033,0.021367,NaN,0.138357,0.082688,1.250862e+09,1.075748e+09,259574000.0,0.100915,4.818902,4.144283,2.537331,0.640721,0.067918,0.327050,0.0,0.721909,0.772748,0
3,cash_burn_case,1,3 - Leveraged / elevated risk profile,3,0.303008,0.157456,0.481975,0.0,2 - Good credit profile,1.273883,4.0,4 - Weak credit profile,0.702223,Neutral,Company remains materially closer to its assig...,56.714265,1.0,"interest_coverage_below_1, negative_cfo_to_assets",High caution,"elevated_debt_to_assets, elevated_debt_to_ebit...",The company is assigned to 3 - Leveraged / ele...,The cluster assignment appears primarily suppo...,"The model output may be directionally useful, ...",21.478318,0.784894,0.471697,2.192857,0.215106,0.052770,-0.017295,-0.124672,1.208901,1.893278,1.068811,0.537243,-0.476224,0.021367,NaN,-0.158839,0.082688,1.003640e+09,8.913610e+08,259574000.0,0.100915,3.866489,3.433938,2.537331,0.488441,0.117137,0.672950,1.0,0.804416,1.000000,0
4,near_default_stress,2,5 - Distressed / near-default proxy,5,0.243697,0.243697,1.282745,4.0,4 - Weak credit profile,1.6

In [12]:
guardrail_cols = [
    "guardrail_level",
    "guardrail_flags",
    "guardrail_summary",
    "analyst_interpretation",
    "commercial_conclusion",
]

display(scored_manual_with_outlook[guardrail_cols])

,guardrail_level,guardrail_flags,guardrail_summary,analyst_interpretation,commercial_conclusion
0,High caution,"elevated_debt_to_assets, elevated_debt_to_ebit...",The model assigns the company to the strongest...,The cluster assignment appears primarily suppo...,"The model output may be directionally useful, ..."


In [13]:
pdf_path = save_credit_pdf_report(
    report_tables=report_tables,
    scored_manual_with_outlook=scored_manual_with_outlook,
    comparison_to_cluster=comparison_to_cluster,
    scored_scenarios=scored_scenarios,
    artifact=artifact,
    output_path=OUTPUT_PATH,
    base_filename=current_file_name,
)

print("PDF credit report saved to:", pdf_path)

PDF credit report saved to: D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\machine_learning\08_final_project_3\outputs\company_scores\EEEC_2025_credit_report.pdf


## 6. Optional: inspect saved artifact internals

In [14]:
# ---------------------------------------------------------------------
# Optional artifact inspection
# ---------------------------------------------------------------------

nonfin_artifact = get_segment_artifact(
    artifact,
    segment=SCORING_SEGMENT,
)

print("Artifact version:")
print(artifact.get("artifact_version"))

print("\nPrimary segment:")
print(artifact.get("primary_segment"))

print("\nFeatures used by model:")
print(nonfin_artifact.get("feature_cols"))

print("\nCluster labels:")
print(nonfin_artifact.get("cluster_labels"))

print("\nRisk rank:")
print(nonfin_artifact.get("risk_rank"))

print("\nCluster sizes:")
display(nonfin_artifact.get("cluster_sizes"))

print("\nCluster profile:")
display(nonfin_artifact.get("cluster_profile"))

print("\nCluster profile ranked:")
display(nonfin_artifact.get("cluster_profile_ranked"))

Artifact version:
v3_scorecard

Primary segment:
Non-financial

Features used by model:
['structural_distress_risk', 'earnings_risk', 'operating_cashflow_risk', 'liquidity_risk', 'leverage_risk', 'debt_service_risk']

Cluster labels:
{3: '1 - Strong relative credit profile', 0: '2 - Good credit profile', 1: '3 - Leveraged / elevated risk profile', 4: '4 - Weak credit profile', 2: '5 - Distressed / near-default proxy'}

Risk rank:
{3: 1, 0: 2, 1: 3, 4: 4, 2: 5}

Cluster sizes:


,cluster,issuer_years,issuers
0,0,4341,1241
1,1,6593,2032
2,2,3350,1572
3,3,7087,1949
4,4,2632,1269



Cluster profile:


,scorecard_credit_score,structural_distress_risk,earnings_risk,operating_cashflow_risk,liquidity_risk,leverage_risk,debt_service_risk,liabilities_risk,debt_load_risk,equity_buffer_risk,cash_buffer_risk,current_liquidity_risk,quick_liquidity_risk,profitability_risk,cashflow_risk,coverage_risk,fcf_risk,ebitda_margin_risk,debt_to_ebitda_risk,net_debt_to_ebitda_risk,ebitda_coverage_risk,negative_ebitda_flag,negative_equity_flag,liabilities_exceed_assets_flag,log_assets,small_company_flag,mid_company_flag,large_company_flag,total_debt,net_debt,ebitda,liabilities_to_assets,debt_to_assets,debt_to_equity,equity_to_assets,cash_to_assets,net_income_to_assets,cfo_to_assets,revenue_to_assets,current_ratio,quick_ratio,interest_coverage,fcf_to_debt,operating_margin,gross_margin,cfo_to_liabilities,capex_to_revenue,ebitda_margin,debt_to_ebitda,net_debt_to_ebitda,ebitda_interest_coverage,debt_service_risk_legacy
cluster,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0,32.468255,0.0,0.165341,0.00000,0.737445,0.266252,0.210794,0.401415,0.074700,0.246633,0.771301,0.791910,0.865893,0.165341,0.00000,0.0,0.000000,0.071443,0.343185,0.389702,0.0,0.0,0.0,0.0,22.805765,0.0,0.0,1.0,2.111126e+09,1.506850e+09,707542500.0,0.670778,0.294820,0.858678,0.301347,0.030583,0.033466,0.081283,0.491363,1.010112,0.350580,3.272810,0.202893,0.128044,0.329196,0.123462,0.025668,0.185711,3.372739,2.863957,5.415796,0.102046
1,58.333333,0.0,1.000000,1.00000,0.000000,0.039044,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.00000,1.0,1.000000,1.000000,1.000000,0.713506,1.0,1.0,0.0,0.0,18.683641,0.0,0.0,1.0,2.258896e+07,-6.722000e+06,-27983250.0,0.299129,0.126675,0.259263,0.600237,0.277374,-0.258140,-0.198351,0.240246,4.310476,2.130207,-27.678366,-1.230223,-0.818926,0.406889,-0.745634,0.028211,-0.697438,9.780351,3.997271,-23.494511,1.000000
2,77.597603,1.0,1.000000,1.00000,0.709803,0.693849,1.000000,1.000000,0.243899,1.000000,0.000000,0.922982,0.808580,1.000000,1.00000,1.0,1.000000,1.000000,1.000000,1.000000,1.0,1.0,1.0,1.0,18.060449,0.0,0.0,1.0,3.047650e+07,8.512271e+06,-10741683.0,1.086288,0.396340,-0.805166,-0.209223,0.103684,-0.277459,-0.151918,0.451918,0.846273,0.393565,-7.905898,-0.521353,-0.529271,0.331066,-0.150882,0.011156,-0.459393,9.587202,6.870450,-6.715322,1.000000
3,11.200344,0.0,0.009368,0.00000,0.067104,0.071860,0.087585,0.006860,0.000000,0.000000,0.000000,0.000000,0.000000,0.009368,0.00000,0.0,0.000000,0.259325,0.094019,0.000000,0.0,0.0,0.0,0.0,21.193320,0.0,0.0,1.0,5.373860e+08,2.252195e+08,123142500.0,0.453773,0.235761,0.482906,0.485135,0.117420,0.049063,0.093812,0.610141,2.365987,1.275307,7.003767,0.280442,0.107985,0.376434,0.202783,0.021270,0.148135,2.376074,1.251290,10.197074,0.000000
4,62.536025,0.0,1.000000,0.80405,0.666973,0.289433,0.904431,0.367144,0.000000,0.259848,0.568291,0.747823,0.798204,1.000000,0.80405,1.0,0.964537,1.000000,1.000000,1.000000,1.0,1.0,0.0,0.0,19.570780,0.0,0.0,1.0,1.402670e+08,6.677050e+07,-4183161.5,0.651929,0.244138,0.837096,0.296061,0.048854,-0.068130,-0.008446,0.458308,1.065221,0.401347,-2.720396,-0.091134,-0.116823,0.295145,-0.029930,0.013599,-0.061367,9.125257,7.786122,-1.439512,0.911562



Cluster profile ranked:


,financial_flag,cluster,issuer_years,issuers,median_log_assets,median_liabilities_to_assets,median_debt_to_assets,median_debt_to_equity,median_equity_to_assets,median_cash_to_assets,median_net_income_to_assets,median_cfo_to_assets,median_revenue_to_assets,median_current_ratio,median_quick_ratio,median_interest_coverage,median_fcf_to_debt,median_operating_margin,median_gross_margin,median_ebitda_margin,median_debt_to_ebitda,median_net_debt_to_ebitda,median_ebitda_interest_coverage,median_leverage_risk,median_liquidity_risk,median_earnings_risk,median_operating_cashflow_risk,median_debt_service_risk,median_structural_distress_risk,median_scorecard_credit_score,rating_style_rank,rating_style_label
3,Non-financial,3,7087,1949,21.193320,0.453773,0.235761,0.482906,0.485135,0.117420,0.049063,0.093812,0.610141,2.365987,1.275307,7.003767,0.280442,0.107985,0.376434,0.148135,2.376074,1.251290,10.197074,0.071860,0.067104,0.009368,0.00000,0.087585,0.0,11.200344,1,1 - Strong relative credit profile
0,Non-financial,0,4341,1241,22.805765,0.670778,0.294820,0.858678,0.301347,0.030583,0.033466,0.081283,0.491363,1.010112,0.350580,3.272810,0.202893,0.128044,0.329196,0.185711,3.372739,2.863957,5.415796,0.266252,0.737445,0.165341,0.00000,0.210794,0.0,32.468255,2,2 - Good credit profile
1,Non-financial,1,6593,2032,18.683641,0.299129,0.126675,0.259263,0.600237,0.277374,-0.258140,-0.198351,0.240246,4.310476,2.130207,-27.678366,-1.230223,-0.818926,0.406889,-0.697438,9.780351,3.997271,-23.494511,0.039044,0.000000,1.000000,1.00000,1.000000,0.0,58.333333,3,3 - Leveraged / elevated risk profile
4,Non-financial,4,2632,1269,19.570780,0.651929,0.244138,0.837096,0.296061,0.048854,-0.068130,-0.008446,0.458308,1.065221,0.401347,-2.720396,-0.091134,-0.116823,0.295145,-0.061367,9.125257,7.786122,-1.439512,0.289433,0.666973,1.000000,0.80405,0.904431,0.0,62.536025,4,4 - Weak credit profile
2,Non-financial,2,3350,1572,18.060449,1.086288,0.396340,-0.805166,-0.209223,0.103684,-0.277459,-0.151918,0.451918,0.846273,0.393565,-7.905898,-0.521353,-0.529271,0.331066,-0.459393,9.587202,6.870450,-6.715322,0.693849,0.709803,1.000000,1.00000,1.000000,1.0,77.597603,5,5 - Distressed / near-default proxy
